# Relative PSF-Flux Light Curves — Colour-coded by Detector (CCD)

This notebook is a variant of `04c_relativeFlux_withfp.ipynb`.

The key difference is the **colour convention**: instead of colouring points by
photometric band, each point is coloured by its **detector number** (CCD ID stored
in column `r:detector`).  A discrete palette (`tab20` + `tab20b`) is used so that
different CCDs can be visually separated even when multiple detectors appear on the
same light curve.

### Scientific motivation

For a photometrically stable star, the relative flux `psfFlux / median(psfFlux)`
should be ≈ 1 at all epochs.  Systematic deviations can originate from:

1. **Detector-to-detector calibration offsets** — if a given CCD consistently
   produces high or low ratios relative to the others, the problem is detector-level
   (flat-field, gain, response non-uniformity).
2. **Time-variable calibration** — if the deviations are correlated with epoch
   (regardless of which detector), the problem is in the time-variable photometric
   solution (zeropoint drift, atmospheric throughput, etc.).

By tagging each measurement with its detector ID, this notebook helps distinguish
between these two hypotheses.

### Data sources
| File | Content |
|---|---|
| `data_FINK_BLOCK_LC_03/src_joined_butler.parquet` | Alert-stream sources (src) — contains `r:detector` |
| `data_FINK_BLOCK_LC_01/gaia_star_stable_fp.parquet` | Forced photometry (fp) |

### Visual convention
| Data type | Marker | Colour |
|---|---|---|
| src | filled circle `o` | detector colour (discrete palette) |
| fp  | hollow circle `o` | detector colour (edge only, no fill) |

Each band subplot shows a **legend of detector IDs** present in that band.
The combined subplot 7 (all bands) also shows the detector legend.

### Subplot layout
`u | g | r | i | z | y | all-bands`  (7 panels per figure)

---

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS
- **Creation date:** 2026-04-06
- **Based on:** `04c_relativeFlux_withfp.ipynb`
- **Subject:** Fink alert broker — Rubin LSST detector-level calibration diagnostics

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend when ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_04c02"
DIR_DATA_SRC = "data_FINK_BLOCK_LC_03"  # src alert data (notebook 03)
DIR_DATA_FP = "data_FINK_BLOCK_LC_01"  # forced photometry (notebook 01)
DIR_FIGS = f"figs_{NB_TAG}"  # output figures
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Source files ──────────────────────────────────────────────────────────────
FILE_BUTLER = os.path.join(DIR_DATA_SRC, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_SRC, "src_joined_consdb.parquet")
FILE_FP = os.path.join(DIR_DATA_FP, "gaia_star_stable_fp.parquet")

# ── Number of top-ranked DIA objects to plot ──────────────────────────────────
N_TOP = 20  # <── change here

# ── Band definitions (for subplot titles only — no band colouring) ────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Column names ──────────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_DET = "r:detector"  # CCD / detector number

FP_COL_OBJ = "diaObjectId"
FP_COL_MJD = "midpointMjdTai"
FP_COL_BAND = "band"
FP_COL_PSF = "psfFlux"
FP_COL_PSFERR = "psfFluxErr"
# fp data may or may not have a detector column — handled gracefully below
FP_COL_DET = None  # set later if the column is found

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Src input directory : {os.path.abspath(DIR_DATA_SRC)}")
print(f"FP  input directory : {os.path.abspath(DIR_DATA_FP)}")
print(f"Figure directory    : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP               : {N_TOP}")

## 2. Load src data (alert stream)

In [ ]:
if os.path.exists(FILE_BUTLER):
    df_src = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file: {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df_src = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found. "
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")

## 3. Load forced photometry data

In [ ]:
if not os.path.exists(FILE_FP):
    raise FileNotFoundError(
        f"Forced photometry file not found: {FILE_FP}\n"
        "Run notebook 01 to produce gaia_star_stable_fp.parquet first."
    )

df_fp_all = pd.read_parquet(FILE_FP)
print(f"Forced photometry file: {FILE_FP}")
print(f"Shape: {df_fp_all.shape[0]:,} rows × {df_fp_all.shape[1]} columns")

## 4. Schema inspection & column auto-detection

In [ ]:
print("=== SRC columns ===")
print(df_src.dtypes.to_string())

In [ ]:
print("=== FP columns ===")
print(df_fp_all.dtypes.to_string())

In [ ]:
# ── Validate src columns ──────────────────────────────────────────────────────
required_src = [COL_OBJ, COL_MJD, COL_BAND, COL_PSF, COL_PSFERR]
missing_src = [c for c in required_src if c not in df_src.columns]
if missing_src:
    raise KeyError(f"Missing required src columns: {missing_src}")

# Detector column must exist in src (comes from Butler join)
if COL_DET not in df_src.columns:
    raise KeyError(
        f"Detector column '{COL_DET}' not found in src. "
        "This notebook requires Butler-joined data (src_joined_butler.parquet)."
    )

print("All required src columns present.")
print(f"Unique CCDs in src : {df_src[COL_DET].nunique()}")
print(f"CCD ID sample      : {sorted(df_src[COL_DET].dropna().unique()[:15].tolist())}")

# ── Auto-detect fp column names ───────────────────────────────────────────────
fp_cols = df_fp_all.columns.tolist()


def _find_col(candidates, available, label):
    for c in candidates:
        if c in available:
            return c
    raise KeyError(
        f"Cannot find {label} column in fp file. Candidates tried: {candidates}. Available: {available[:20]}"
    )


FP_COL_OBJ = _find_col(["diaObjectId", "r:diaObjectId", "objectId"], fp_cols, "diaObjectId")
FP_COL_MJD = _find_col(["midpointMjdTai", "r:midpointMjdTai", "mjd", "MJD"], fp_cols, "midpointMjdTai")
FP_COL_BAND = _find_col(["band", "r:band", "filter"], fp_cols, "band")
FP_COL_PSF = _find_col(["psfFlux", "r:psfFlux"], fp_cols, "psfFlux")
FP_COL_PSFERR = _find_col(["psfFluxErr", "r:psfFluxErr"], fp_cols, "psfFluxErr")

# Detector in fp (optional — may not be present)
for _c in ["r:detector", "detector", "ccd", "CCD"]:
    if _c in fp_cols:
        FP_COL_DET = _c
        print(f"FP detector column found: '{FP_COL_DET}'")
        break
else:
    FP_COL_DET = None
    print("FP detector column NOT found — fp points will use a default grey colour.")

print(f"\nFP column mapping:")
print(f"  diaObjectId    → {FP_COL_OBJ}")
print(f"  midpointMjdTai → {FP_COL_MJD}")
print(f"  band           → {FP_COL_BAND}")
print(f"  psfFlux        → {FP_COL_PSF}")
print(f"  psfFluxErr     → {FP_COL_PSFERR}")
print(f"  detector       → {FP_COL_DET}")

## 5. Build detector colour palette

A discrete palette is constructed from the detector IDs present in the src data.
The most frequently observed CCDs (those that received the most alerts — typically
the central detectors of the focal plane) are assigned the most visually distinct
colours at the beginning of the `tab20` + `tab20b` palette.

The palette is built globally once so that the same detector always maps to the
same colour across all objects and all band subplots.

In [ ]:
# ── Sort detectors by decreasing alert count → most active CCDs get first colours
det_counts = df_src[COL_DET].value_counts()  # descending by default
all_det_ids = det_counts.index.tolist()  # sorted: most active first
n_det = len(all_det_ids)
print(f"Total unique detector IDs in src : {n_det}")
print(f"Top-10 most active CCDs (by alert count):")
print(det_counts.head(10).to_string())

# ── Build palette: tab20 (20 colours) + tab20b (20 colours) → 40 colours
# If more than 40 detectors, cycle with reduced alpha
tab20_colors = [mcm.tab20(i) for i in np.linspace(0, 1, 20, endpoint=False)]
tab20b_colors = [mcm.tab20b(i) for i in np.linspace(0, 1, 20, endpoint=False)]
palette_base = tab20_colors + tab20b_colors  # 40 distinct colours

DET_PALETTE = {}
for i, det_id in enumerate(all_det_ids):
    DET_PALETTE[det_id] = palette_base[i % len(palette_base)]

print(f"\nPalette built: {len(palette_base)} base colours for {n_det} detectors.")
if n_det > len(palette_base):
    print(f"  WARNING: {n_det - len(palette_base)} detectors will share colours (cycled).")

In [ ]:
# ── Optional: visual check of the palette (top-40 CCDs) ──────────────────────
n_show = min(40, n_det)
fig_pal, ax_pal = plt.subplots(figsize=(14, 2.2), constrained_layout=True)
for i, det_id in enumerate(all_det_ids[:n_show]):
    ax_pal.add_patch(plt.Rectangle((i, 0), 1, 1, color=DET_PALETTE[det_id]))
    ax_pal.text(
        i + 0.5, 0.5, str(det_id), ha="center", va="center", fontsize=6.5, color="k", fontweight="bold"
    )
ax_pal.set_xlim(0, n_show)
ax_pal.set_ylim(0, 1)
ax_pal.set_xticks([])
ax_pal.set_yticks([])
ax_pal.set_title(
    f"Detector colour palette — top-{n_show} CCDs sorted by alert count (left = most active)",
    fontsize=10,
)
savefig("detector_colour_palette")
plt.show()

## 6. Rank DIA objects by number of src alerts (decreasing)

In [ ]:
alert_counts = df_src.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects in src : {len(alert_counts):,}")
print(f"Top {N_TOP} by alert count:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df_src[df_src[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

# Filter fp to top-N objects (type-safe)
fp_obj_ids = df_fp_all[FP_COL_OBJ].unique()
try:
    top_objects_fp = [type(fp_obj_ids[0])(x) for x in top_objects]
except Exception:
    top_objects_fp = top_objects

df_fp = df_fp_all[df_fp_all[FP_COL_OBJ].isin(top_objects_fp)].copy()
df_fp[FP_COL_MJD] = df_fp[FP_COL_MJD].astype(float)

print(f"Src rows for top-{N_TOP} objects : {len(df_top):,}")
print(f"FP  rows for top-{N_TOP} objects : {len(df_fp):,}")

## 7. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert MJD floats to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def compute_src_median_and_ratio(flux, flux_err):
    """
    Compute flux / median(flux) and propagate errors.

    Returns
    -------
    ratio, ratio_err, f_med, sigma_ratio
    """
    f = np.asarray(flux, dtype=float)
    fe = np.asarray(flux_err, dtype=float)
    finite_mask = np.isfinite(f) & (f > 0)
    if finite_mask.sum() == 0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), np.nan, np.nan
    f_med = float(np.median(f[finite_mask]))
    if f_med == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan), 0.0, np.nan
    ratio = f / f_med
    ratio_err = fe / f_med
    sigma_ratio = float(np.nanstd(ratio[finite_mask]))
    return ratio, ratio_err, f_med, sigma_ratio


def normalise_fp(fp_flux, fp_flux_err, f_med_src):
    """Normalise fp fluxes using the src median for the same (object, band)."""
    f = np.asarray(fp_flux, dtype=float)
    fe = np.asarray(fp_flux_err, dtype=float)
    if np.isnan(f_med_src) or f_med_src == 0.0:
        return np.full_like(f, np.nan), np.full_like(fe, np.nan)
    return f / f_med_src, fe / f_med_src


def _add_date_axis(ax, dt_array, t0_mjd):
    """Add a secondary top x-axis with calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def det_color(det_id):
    """Return the colour associated with a detector ID."""
    return DET_PALETTE.get(det_id, "#888888")  # grey fallback


def build_det_legend_handles(det_ids_present):
    """
    Build matplotlib legend handles for a set of detector IDs.

    Parameters
    ----------
    det_ids_present : iterable of detector IDs (int or similar)

    Returns
    -------
    List of matplotlib Line2D objects suitable for ax.legend(handles=...).
    """
    from matplotlib.lines import Line2D

    handles = []
    for det_id in sorted(det_ids_present):
        c = det_color(det_id)
        h = Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=c,
            markeredgecolor=c,
            markersize=5,
            label=f"CCD {det_id}",
        )
        handles.append(h)
    return handles


print("Utility functions defined.")

## 8. Main plotting function: src + fp relative flux, colour by detector

For each measurement the colour is determined by `r:detector` (src) or
`FP_COL_DET` (fp, if available).  Within each band subplot a legend lists
the detector IDs that actually appear in that band's data.  Subplot 7
(all-bands combined) carries a full detector legend.

### Notation in legends
- `CCD N src` — src point from detector N
- `CCD N fp`  — forced-photometry point from detector N

In [ ]:
def plot_src_fp_object_by_detector(obj_id, df_src_obj, df_fp_obj, axes, yscale_min=0.5, yscale_max=1.5):
    """
    Plot relative PSF-flux light curves (src + fp) for one DIA object,
    with each point coloured by its detector (CCD) number.

    Parameters
    ----------
    obj_id     : int / str  — diaObjectId
    df_src_obj : DataFrame  — src rows for this object
    df_fp_obj  : DataFrame  — fp rows for this object (may be empty)
    axes       : list of 7 Axes — [u, g, r, i, z, y, all-bands]

    Returns
    -------
    t0_date : str  — ISO date of the first src alert
    """
    t0_mjd = float(df_src_obj[COL_MJD].min())
    t0_date = mjd_to_datestr([t0_mjd])[0]

    all_dt_src = df_src_obj[COL_MJD].values - t0_mjd
    all_dt_fp = (df_fp_obj[FP_COL_MJD].values - t0_mjd) if len(df_fp_obj) > 0 else np.array([])
    all_dt_combined = np.concatenate([all_dt_src, all_dt_fp])
    # common xlim boundaries
    all_dt_combined_min = all_dt_combined.min() - 1.0
    all_dt_combined_max = all_dt_combined.max() + 1.0

    # Pre-compute per-band src median (needed for fp normalisation)
    src_median_per_band = {}

    # Keep track of detectors plotted on each axis for legend building
    dets_per_ax = {i: set() for i in range(len(axes))}

    # ── Per-band subplots (0..5) ──────────────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        bcolor = BAND_COLORS[band]  # used only for the subplot title

        # ---- SRC ----
        mask_src = df_src_obj[COL_BAND] == band
        df_b_src = df_src_obj[mask_src].sort_values(COL_MJD)

        if len(df_b_src) == 0:
            src_median_per_band[band] = np.nan
            ax.text(
                0.5,
                0.5,
                "no src data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9, color=bcolor)
            continue

        dt_src = df_b_src[COL_MJD].values - t0_mjd
        src_ratio, src_ratio_err, f_med, sigma_src = compute_src_median_and_ratio(
            df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
        )
        src_median_per_band[band] = f_med

        # Plot one point at a time to assign individual detector colours
        for i_pt in range(len(df_b_src)):
            det_id = df_b_src[COL_DET].iloc[i_pt]
            c = det_color(det_id)
            dets_per_ax[idx].add(det_id)
            dets_per_ax[6].add(det_id)  # also collect for all-bands subplot
            ax.errorbar(
                dt_src[i_pt],
                src_ratio[i_pt],
                yerr=src_ratio_err[i_pt],
                fmt="o",
                ms=4,
                color=c,
                ecolor=c,
                elinewidth=0.8,
                capsize=2,
                alpha=0.85,
            )

        # ---- FP ----
        dt_all_band = dt_src
        if len(df_fp_obj) > 0 and not np.isnan(f_med):
            mask_fp = df_fp_obj[FP_COL_BAND] == band
            df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)

            if len(df_b_fp) > 0:
                dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                fp_ratio, fp_ratio_err = normalise_fp(
                    df_b_fp[FP_COL_PSF].values,
                    df_b_fp[FP_COL_PSFERR].values,
                    f_med,
                )
                dt_all_band = np.concatenate([dt_src, dt_fp])

                for i_pt in range(len(df_b_fp)):
                    # Use fp detector if available, else mirror the band colour
                    if FP_COL_DET is not None and FP_COL_DET in df_b_fp.columns:
                        det_id_fp = df_b_fp[FP_COL_DET].iloc[i_pt]
                    else:
                        det_id_fp = None
                    c_fp = det_color(det_id_fp) if det_id_fp is not None else "#888888"
                    if det_id_fp is not None:
                        dets_per_ax[idx].add(det_id_fp)
                        dets_per_ax[6].add(det_id_fp)
                    ax.errorbar(
                        dt_fp[i_pt],
                        fp_ratio[i_pt],
                        yerr=fp_ratio_err[i_pt],
                        fmt="o",
                        ms=5,
                        color="none",
                        mec=c_fp,
                        mew=1.2,
                        ecolor=c_fp,
                        elinewidth=0.8,
                        capsize=2,
                        alpha=0.80,
                    )

        # force a common x-scale
        ax.set_xlim(all_dt_combined_min, all_dt_combined_max)
        ax.set_ylim(yscale_min, yscale_max)

        ax.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
        ax.set_title(
            f"band {band}  |  σ_src={sigma_src:.3f}",
            fontsize=9,
            color=bcolor,
        )
        ax.set_ylabel("psfFlux / median(src)", fontsize=8)

        # Detector legend for this band
        if dets_per_ax[idx]:
            handles = build_det_legend_handles(dets_per_ax[idx])
            ax.legend(
                handles=handles,
                title="CCD",
                title_fontsize=7,
                fontsize=6,
                loc="best",
                ncol=max(1, len(dets_per_ax[idx]) // 8),
            )

        _add_date_axis(ax, dt_all_band, t0_mjd)

    # ── Subplot 7: all bands combined, colour by detector ─────────────────────
    ax_all = axes[-1]

    for band in BANDS:
        f_med = src_median_per_band.get(band, np.nan)

        # src
        mask_src = df_src_obj[COL_BAND] == band
        df_b_src = df_src_obj[mask_src].sort_values(COL_MJD)
        if len(df_b_src) > 0 and not np.isnan(f_med):
            dt_src = df_b_src[COL_MJD].values - t0_mjd
            src_ratio, src_ratio_err, _, sigma_src = compute_src_median_and_ratio(
                df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
            )
            for i_pt in range(len(df_b_src)):
                det_id = df_b_src[COL_DET].iloc[i_pt]
                c = det_color(det_id)
                ax_all.errorbar(
                    dt_src[i_pt],
                    src_ratio[i_pt],
                    yerr=src_ratio_err[i_pt],
                    fmt="o",
                    ms=3,
                    color=c,
                    ecolor=c,
                    elinewidth=0.7,
                    capsize=2,
                    alpha=0.75,
                )

        # fp
        if len(df_fp_obj) > 0 and not np.isnan(f_med):
            mask_fp = df_fp_obj[FP_COL_BAND] == band
            df_b_fp = df_fp_obj[mask_fp].sort_values(FP_COL_MJD)
            if len(df_b_fp) > 0:
                dt_fp = df_b_fp[FP_COL_MJD].values - t0_mjd
                fp_ratio, fp_ratio_err = normalise_fp(
                    df_b_fp[FP_COL_PSF].values,
                    df_b_fp[FP_COL_PSFERR].values,
                    f_med,
                )
                for i_pt in range(len(df_b_fp)):
                    if FP_COL_DET is not None and FP_COL_DET in df_b_fp.columns:
                        det_id_fp = df_b_fp[FP_COL_DET].iloc[i_pt]
                    else:
                        det_id_fp = None
                    c_fp = det_color(det_id_fp) if det_id_fp is not None else "#888888"
                    ax_all.errorbar(
                        dt_fp[i_pt],
                        fp_ratio[i_pt],
                        yerr=fp_ratio_err[i_pt],
                        fmt="o",
                        ms=4,
                        color="none",
                        mec=c_fp,
                        mew=1.2,
                        ecolor=c_fp,
                        elinewidth=0.8,
                        capsize=2,
                        alpha=0.70,
                    )

    # force a common x-scale,y-scale
    ax_all.set_xlim(all_dt_combined_min, all_dt_combined_max)
    ax_all.set_ylim(yscale_min, yscale_max)

    ax_all.axhline(1.0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax_all.set_title("all bands — src (●) + fp (○)\ncolour = detector (CCD)", fontsize=9)
    ax_all.set_ylabel("psfFlux / median(src)", fontsize=8)

    # Detector legend for all-bands subplot
    if dets_per_ax[6]:
        handles_all = build_det_legend_handles(dets_per_ax[6])
        ax_all.legend(
            handles=handles_all,
            title="CCD",
            title_fontsize=7,
            fontsize=6,
            loc="best",
            ncol=max(1, len(dets_per_ax[6]) // 8),
        )

    _add_date_axis(ax_all, all_dt_combined, t0_mjd)

    return t0_date


print("plot_src_fp_object_by_detector() defined.")

## 9. Main loop: plot all top-N objects

In [ ]:
n_subplots = len(BANDS) + 1  # 6 bands + combined

for rank, obj_id in enumerate(top_objects):
    # src rows
    df_src_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_src_obj)

    # fp rows (type-safe match)
    try:
        fp_obj_id = type(df_fp_all[FP_COL_OBJ].iloc[0])(obj_id)
    except Exception:
        fp_obj_id = obj_id
    df_fp_obj = df_fp[df_fp[FP_COL_OBJ] == fp_obj_id].sort_values(FP_COL_MJD)
    n_fp = len(df_fp_obj)

    # Field name
    field_vals = df_src_obj["field"].dropna().unique() if "field" in df_src_obj.columns else []
    field_str = field_vals[0] if len(field_vals) > 0 else "?"

    # Unique detectors seen for this object
    det_list = sorted(df_src_obj[COL_DET].dropna().unique().tolist())
    n_det_obj = len(det_list)

    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 5.0),
        sharey=False,
        constrained_layout=True,
    )

    t0_date = plot_src_fp_object_by_detector(obj_id, df_src_obj, df_fp_obj, axes)

    for ax in axes:
        ax.set_xlabel("Δt [days] from first src alert", fontsize=8)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field_str}  |  "
        f"N_src={n_alerts}  N_fp={n_fp}  N_CCD={n_det_obj}  |  t₀={t0_date}\n"
        "src ● (filled)   fp ○ (hollow)   —  colour = detector (CCD number)  "
        "—  normalised by median(src psfFlux)",
        fontsize=10,
        y=1.05,
    )

    savefig(f"relflux_psf_srcfp_bydet_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()

## 10. Summary table: per-object, per-band σ(ratio) + detector census

Reports:
- `sigma_src_{band}` — RMS scatter of the src relative flux ratio per band
- `n_fp_{band}`      — number of forced-photometry points per band
- `n_det_{band}`     — number of distinct CCDs contributing src points in each band
- `det_ids`          — comma-separated list of all CCD IDs seen for this object

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_src_obj = df_top[df_top[COL_OBJ] == obj_id]
    n_total = len(df_src_obj)
    t0_mjd = float(df_src_obj[COL_MJD].min())
    t0_date = mjd_to_datestr([t0_mjd])[0]

    try:
        fp_obj_id = type(df_fp_all[FP_COL_OBJ].iloc[0])(obj_id)
    except Exception:
        fp_obj_id = obj_id
    df_fp_obj = df_fp[df_fp[FP_COL_OBJ] == fp_obj_id]

    all_dets = sorted(df_src_obj[COL_DET].dropna().unique().tolist())

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_src": n_total,
        "n_fp": len(df_fp_obj),
        "t0_date": t0_date,
        "det_ids": ",".join(str(d) for d in all_dets),
        "n_det_total": len(all_dets),
    }

    for band in BANDS:
        df_b_src = df_src_obj[df_src_obj[COL_BAND] == band]
        if len(df_b_src) == 0:
            row[f"sigma_src_{band}"] = np.nan
            row[f"n_fp_{band}"] = 0
            row[f"n_det_{band}"] = 0
            continue

        _, _, _, sigma_src = compute_src_median_and_ratio(
            df_b_src[COL_PSF].values, df_b_src[COL_PSFERR].values
        )
        row[f"sigma_src_{band}"] = round(sigma_src, 4)
        row[f"n_fp_{band}"] = int((df_fp_obj[FP_COL_BAND] == band).sum()) if len(df_fp_obj) > 0 else 0
        row[f"n_det_{band}"] = int(df_b_src[COL_DET].nunique())

    records.append(row)

df_summary = pd.DataFrame(records)
print("Summary table — relative-flux scatter + fp counts + detector census:")
display(df_summary)

In [ ]:
summary_path = os.path.join(DIR_FIGS, f"sigma_ratio_bydet_summary_{src_label}.csv")
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to {summary_path}")

## 11. Interpretation guide

### How to distinguish detector effects from time-variable calibration

Look at each band subplot and the all-bands subplot:

| Observation pattern | Most likely explanation |
|---|---|
| Points of the **same colour (CCD)** are consistently **above or below 1** across multiple epochs | Detector-level calibration offset (flat-field, gain, response) |
| Points of **different colours scatter together** at the same epoch | Time-variable photometric zeropoint (atmosphere, throughput drift) |
| **Isolated excursions** by a single CCD at a specific epoch | Either a bad exposure on that CCD, or a transient calibration failure |
| The **fp hollow circles** and **src filled circles** of the same CCD agree | Consistent PSF model; disagreement may indicate a PSF-model issue for that detector |

### What to do next
- If CCD-correlated offsets are found: investigate the flat-field or gain tables for
  the affected detectors in `lsst_distrib`.
- If time-correlated offsets are found: compare with the photometric zeropoint
  stored in ConsDB (`psfMagLim`, `skyBg`) for the corresponding visits.
- The `det_ids` column in the summary table above identifies which CCDs contribute
  to each object — useful for targeted Butler queries.